In [6]:
import os

print("=== What's in /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files[:10]:  # limit files shown
        print(f"{subindent}{f}")
    if len(files) > 10:
        print(f"{subindent}... and {len(files)-10} more files")

=== What's in /kaggle/input ===
input/
  datasets/
    ekrasafdar/
      brain-tumor-mri/
        Training/
          pituitary/
            Tr-pi_124.jpg
            Tr-pi_949.jpg
            Tr-pi_786.jpg
            Tr-pi_371.jpg
            Tr-pi_599.jpg
            Tr-pi_802.jpg
            Tr-pi_1323.jpg
            Tr-pi_1347.jpg
            Tr-pi_955.jpg
            Tr-pi_778.jpg
            ... and 1390 more files
          notumor/
            Tr-no_323.jpg
            Tr-no_86.jpg
            Tr-no_737.jpg
            Tr-no_452.jpg
            Tr-no_557.jpg
            Tr-no_757.jpg
            Tr-no_54.jpg
            Tr-no_472.jpg
            Tr-no_1240.jpg
            Tr-no_373.jpg
            ... and 1390 more files
          meningioma/
            Tr-me_166.jpg
            Tr-me_1256.jpg
            Tr-me_115.jpg
            Tr-me_431.jpg
            Tr-me_971.jpg
            Tr-me_410.jpg
            Tr-me_361.jpg
            Tr-me_1293.jpg
            Tr-me_516.jpg
 

In [5]:
import os, zipfile, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mob
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

tf.get_logger().setLevel('ERROR')

# ========== 1. FIND & EXTRACT YOUR DATASET ==========
zip_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.zip'):
            zip_path = os.path.join(root, f)
            print(f"Found zip: {zip_path}")
            break
    if zip_path: break

if not zip_path:
    raise FileNotFoundError("Add your dataset via '+ Add Input' on the right panel")

extract_to = '/kaggle/working/data'
os.makedirs(extract_to, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)

# Find Training/Testing folders (handles nested folders)
train_dir, test_dir = None, None
for root, dirs, files in os.walk(extract_to):
    if 'Training' in dirs and not train_dir:
        train_dir = os.path.join(root, 'Training')
    if 'Testing' in dirs and not test_dir:
        test_dir = os.path.join(root, 'Testing')

print(f"Train: {train_dir}")
print(f"Test:  {test_dir}")

# ========== 2. CONFIG ==========
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 30
LR = 1e-4
SEEDS = [42, 123, 456]
PRE = {'mobilenetv2': pre_mob, 'resnet50': pre_res, 'efficientnetb0': pre_eff}

# ========== 3. MODEL BUILDER ==========
def build(name, seed):
    tf.random.set_seed(seed); np.random.seed(seed)
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'mobilenetv2': base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'resnet50': base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    else: base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
    
    base.trainable = True
    for L in base.layers[:-50]: L.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)
    
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(LR), loss='categorical_crossentropy', metrics=['accuracy'])
    return m

# ========== 4. GENERATORS ==========
def gens(name):
    p = PRE[name]
    tr_aug = ImageDataGenerator(preprocessing_function=p, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    
    tr = tr_aug.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(train_dir, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)
    te = ImageDataGenerator(preprocessing_function=p).flow_from_directory(
        test_dir, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)
    return tr, val, te

# ========== 5. TRAIN & EVALUATE ==========
def run(name):
    print(f"\n{'='*60}\nMODEL: {name.upper()}\n{'='*60}")
    results = []
    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")
        m = build(name, seed)
        tr, val, te = gens(name)
        
        cb = [
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0)
        ]
        m.fit(tr, validation_data=val, epochs=EPOCHS, callbacks=cb, verbose=1)
        
        te.reset(); loss, acc = m.evaluate(te, verbose=0)
        te.reset(); yp = np.argmax(m.predict(te, verbose=0), axis=1)
        yt = te.classes
        f1 = f1_score(yt, yp, average='weighted')
        
        results.append({'seed': seed, 'acc': acc*100, 'f1': f1*100, 'yt': yt, 'yp': yp})
        print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")
    
    A = [r['acc'] for r in results]
    F = [r['f1'] for r in results]
    print(f"\n>>> SUMMARY: {np.mean(A):.2f}% ± {np.std(A):.2f}% | F1: {np.mean(F):.2f}% ± {np.std(F):.2f}%")
    return results

# ========== 6. RUN ALL ==========
all_res = {}
for n in ['mobilenetv2', 'resnet50', 'efficientnetb0']:
    all_res[n] = run(n)

# ========== 7. FINAL TABLE ==========
print("\n\n" + "="*70 + "\nPAPER TABLE — Multi-Seed Results (Mean ± Std)\n" + "="*70)
print(f"{'Model':<20} {'Accuracy':<25} {'F1-Score':<25}")
print("-"*70)
for n, r in all_res.items():
    A = [x['acc'] for x in r]; F = [x['f1'] for x in r]
    print(f"{n:<20} {np.mean(A):.2f}±{np.std(A):.2f}%         {np.mean(F):.2f}±{np.std(F):.2f}%")

# ========== 8. SAVE CONFUSION MATRICES ==========
for n, r in all_res.items():
    best = max(r, key=lambda x: x['acc'])
    cm = confusion_matrix(best['yt'], best['yp'])
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['glioma','meningioma','notumor','pituitary'],
        yticklabels=['glioma','meningioma','notumor','pituitary'])
    plt.title(f'{n.upper()} — Seed {best["seed"]}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{n}_cm.png', dpi=300)
    plt.close()
    print(f"Saved: {n}_cm.png")

print("\n✅ DONE. Copy the table above and download the PNGs from /kaggle/working/")

FileNotFoundError: Add your dataset via '+ Add Input' on the right panel

In [ ]:
import os, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mob
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

tf.get_logger().setLevel('ERROR')

# ========== PATHS (already unzipped) ==========
BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

print(f"Train: {TRAIN_DIR}")
print(f"Test:  {TEST_DIR}")

# ========== CONFIG ==========
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS = 30
LR = 1e-4
SEEDS = [42, 123, 456]
PRE = {'mobilenetv2': pre_mob, 'resnet50': pre_res, 'efficientnetb0': pre_eff}

# ========== MODEL BUILDER ==========
def build(name, seed):
    tf.random.set_seed(seed); np.random.seed(seed)
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'mobilenetv2': base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'resnet50': base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    else: base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
    
    base.trainable = True
    for L in base.layers[:-50]: L.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)
    
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(LR), loss='categorical_crossentropy', metrics=['accuracy'])
    return m

# ========== GENERATORS ==========
def gens(name):
    p = PRE[name]
    tr_aug = ImageDataGenerator(preprocessing_function=p, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    
    tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)
    te = ImageDataGenerator(preprocessing_function=p).flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical', shuffle=False)
    return tr, val, te

# ========== TRAIN & EVALUATE ==========
def run(name):
    print(f"\n{'='*60}\nMODEL: {name.upper()}\n{'='*60}")
    results = []
    for seed in SEEDS:
        print(f"\n--- Seed {seed} ---")
        m = build(name, seed)
        tr, val, te = gens(name)
        
        cb = [
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=0)
        ]
        m.fit(tr, validation_data=val, epochs=EPOCHS, callbacks=cb, verbose=1)
        
        te.reset(); loss, acc = m.evaluate(te, verbose=0)
        te.reset(); yp = np.argmax(m.predict(te, verbose=0), axis=1)
        yt = te.classes
        f1 = f1_score(yt, yp, average='weighted')
        
        results.append({'seed': seed, 'acc': acc*100, 'f1': f1*100, 'yt': yt, 'yp': yp})
        print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")
    
    A = [r['acc'] for r in results]
    F = [r['f1'] for r in results]
    print(f"\n>>> SUMMARY: {np.mean(A):.2f}% ± {np.std(A):.2f}% | F1: {np.mean(F):.2f}% ± {np.std(F):.2f}%")
    return results

# ========== RUN ALL ==========
all_res = {}
for n in ['mobilenetv2', 'resnet50', 'efficientnetb0']:
    all_res[n] = run(n)

# ========== FINAL TABLE ==========
print("\n\n" + "="*70 + "\nPAPER TABLE — Multi-Seed Results (Mean ± Std)\n" + "="*70)
print(f"{'Model':<20} {'Accuracy':<25} {'F1-Score':<25}")
print("-"*70)
for n, r in all_res.items():
    A = [x['acc'] for x in r]; F = [x['f1'] for x in r]
    print(f"{n:<20} {np.mean(A):.2f}±{np.std(A):.2f}%         {np.mean(F):.2f}±{np.std(F):.2f}%")

# ========== SAVE CONFUSION MATRICES ==========
for n, r in all_res.items():
    best = max(r, key=lambda x: x['acc'])
    cm = confusion_matrix(best['yt'], best['yp'])
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['glioma','meningioma','notumor','pituitary'],
        yticklabels=['glioma','meningioma','notumor','pituitary'])
    plt.title(f'{n.upper()} — Seed {best["seed"]}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{n}_cm.png', dpi=300)
    plt.close()
    print(f"Saved: {n}_cm.png")

print("\n✅ DONE. Copy the table above.")

Train: /kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training
Test:  /kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Testing

MODEL: MOBILENETV2

--- Seed 42 ---


/tmp/ipykernel_58/3544690396.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  if name == 'mobilenetv2': base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)


Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30


2026-08-01 06:38:08.609212: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 06:38:08.752625: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 06:38:08.889187: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1785566296.032700     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 122s 652ms/step - accuracy: 0.7612 - loss: 0.6892 - val_accuracy: 0.7089 - val_loss: 0.8549 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 515ms/step - accuracy: 0.8797 - loss: 0.3549 - val_accuracy: 0.7580 - val_loss: 0.7785 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 519ms/step - accuracy: 0.9179 - loss: 0.2263 - val_accuracy: 0.8071 - val_loss: 0.6348 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 517ms/step - accuracy: 0.9379 - loss: 0.1766 - val_accuracy: 0.8732 - val_loss: 0.4215 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 515ms/step - accuracy: 0.9507 - loss: 0.1377 - val_accuracy: 0.9009 - val_loss: 0.3017 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 511ms/step - accuracy: 0.9589 - loss: 0.1186 - val_accuracy: 0.8911 - val_loss: 0.3507 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 517ms/step -

...

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 97s 552ms/step - accuracy: 0.7382 - loss: 0.7609 - val_accuracy: 0.7580 - val_loss: 0.6679 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 519ms/step - accuracy: 0.8808 - loss: 0.3452 - val_accuracy: 0.7955 - val_loss: 0.6316 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 513ms/step - accuracy: 0.9174 - loss: 0.2296 - val_accuracy: 0.7893 - val_loss: 0.6953 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 516ms/step - accuracy: 0.9371 - loss: 0.1812 - val_accuracy: 0.8393 - val_loss: 0.5677 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 511ms/step - accuracy: 0.9478 - loss: 0.1510 - val_accuracy: 0.8857 - val_loss: 0.3675 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 513ms/step - accuracy: 0.9578 - los

/tmp/ipykernel_58/3544690396.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  if name == 'mobilenetv2': base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)


Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 97s 548ms/step - accuracy: 0.7422 - loss: 0.7680 - val_accuracy: 0.7661 - val_loss: 0.5866 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 509ms/step - accuracy: 0.8835 - loss: 0.3455 - val_accuracy: 0.7527 - val_loss: 0.6595 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 508ms/step - accuracy: 0.9223 - loss: 0.2199 - val_accuracy: 0.8732 - val_loss: 0.3908 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 78s 556ms/step - accuracy: 0.9375 - loss: 0.1730 - val_accuracy: 0.8911 - val_loss: 0.3475 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 513ms/step - accuracy: 0.9484 - loss: 0.1423 - val_accuracy: 0.8991 - val_loss: 0.3804 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 510ms/step - accuracy: 0.9607 - los

2026-08-01 10:25:41.792972: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 10:25:41.936299: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 10:25:42.294775: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 10:25:42.435467: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 10:25:43.217044: E external/local_xla/xla/stream_

140/140 ━━━━━━━━━━━━━━━━━━━━ 113s 570ms/step - accuracy: 0.6933 - loss: 0.8867 - val_accuracy: 0.8393 - val_loss: 0.4864 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 518ms/step - accuracy: 0.8364 - loss: 0.4674 - val_accuracy: 0.8946 - val_loss: 0.3014 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 73s 520ms/step - accuracy: 0.8708 - loss: 0.3494 - val_accuracy: 0.9152 - val_loss: 0.2354 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 514ms/step - accuracy: 0.8924 - loss: 0.2906 - val_accuracy: 0.9143 - val_loss: 0.2324 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 513ms/step - accuracy: 0.9121 - loss: 0.2428 - val_accuracy: 0.9402 - val_loss: 0.2058 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 508ms/step - accuracy: 0.9301 - loss: 0.1968 - val_accuracy: 0.9446 - val_loss: 0.1746 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 71s 510ms/step -

In [9]:

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score

BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

tf.random.set_seed(456)
np.random.seed(456)

inp = layers.Input(shape=(224, 224, 3))
base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
base.trainable = True
for L in base.layers[:-50]: L.trainable = False

x = layers.GlobalAveragePooling2D()(base.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
out = layers.Dense(4, activation='softmax')(x)

model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

tr_aug = ImageDataGenerator(preprocessing_function=pre_eff, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
    horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)

tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='training', seed=42)
val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)
te = ImageDataGenerator(preprocessing_function=pre_eff).flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)

cb = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)

te.reset(); loss, acc = model.evaluate(te, verbose=0)
te.reset(); yp = np.argmax(model.predict(te, verbose=0), axis=1)
yt = te.classes
f1 = f1_score(yt, yp, average='weighted')

print(f"\n>>> EFFICIENTNETB0 SEED 456: Accuracy {acc*100:.2f}% | F1 {f1*100:.2f}%")

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 109s 567ms/step - accuracy: 0.6625 - loss: 0.9478 - val_accuracy: 0.8321 - val_loss: 0.4997 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 517ms/step - accuracy: 0.8205 - loss: 0.4968 - val_accuracy: 0.8911 - val_loss: 0.2850 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 518ms/step - accuracy: 0.8676 - loss: 0.3554 - val_accuracy: 0.9107 - val_loss: 0.2313 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 74s 525ms/step - accuracy: 0.8844 - loss: 0.3091 - val_accuracy: 0.9277 - val_loss: 0.1950 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 512ms/step - accuracy: 0.9076 - loss: 0.2452 - val_accuracy: 0.9366 - val_loss: 0.1714 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 72s 514ms/step - accuracy: 0.9230 - lo

In [ ]:
"""
TAB 2 FIXED: MC Dropout + Uncertainty + Benchmarking
No fragile FLOPs calculation. Just params, time, throughput.
"""

import os, time, numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre_mob
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

tf.get_logger().setLevel('ERROR')

# ========== PATHS ==========
BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

# ========== PART 1: BENCHMARKING (Simple & Reliable) ==========
def benchmark_model(model_name):
    """Builds model and measures params + inference time. No FLOPs (fragile)."""
    inp = layers.Input(shape=(224, 224, 3))
    if model_name == 'mobilenetv2': 
        base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif model_name == 'resnet50': 
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    else: 
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)
    model = keras.Model(inp, out)
    
    total_params = model.count_params()
    trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    
    # Inference timing
    sample = tf.random.normal((1, 224, 224, 3))
    _ = model(sample, training=False)  # warmup
    
    times = []
    for _ in range(100):
        t0 = time.time()
        _ = model(sample, training=False)
        times.append(time.time() - t0)
    
    mean_ms = np.mean(times) * 1000
    std_ms = np.std(times) * 1000
    throughput = 1000 / mean_ms  # images per second (batch=1)
    
    return {
        'name': model_name,
        'total_params': total_params,
        'trainable_params': trainable,
        'inference_ms': mean_ms,
        'inference_std': std_ms,
        'throughput': throughput,
    }

print("="*60)
print("PART 1: INFERENCE BENCHMARKING")
print("="*60)

benchmarks = []
for name in ['mobilenetv2', 'resnet50', 'efficientnetb0']:
    print(f"\nBenchmarking {name}...")
    bm = benchmark_model(name)
    benchmarks.append(bm)
    print(f"  Params: {bm['total_params']:,}")
    print(f"  Inference: {bm['inference_ms']:.2f} ± {bm['inference_std']:.2f} ms")
    print(f"  Throughput: {bm['throughput']:.1f} img/s")

print("\n" + "="*60)
print("BENCHMARK TABLE (for paper)")
print("="*60)
print(f"{'Model':<18} {'Params (M)':<14} {'Inference (ms)':<18} {'Throughput':<15}")
print("-"*60)
for bm in benchmarks:
    print(f"{bm['name']:<18} {bm['total_params']/1e6:<14.2f} {bm['inference_ms']:<18.2f} {bm['throughput']:<15.1f}")

# ========== PART 2: MC DROPOUT MODEL ==========
def build_mc_model():
    tf.random.set_seed(42)
    np.random.seed(42)
    
    inp = layers.Input(shape=(224, 224, 3))
    base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    base.trainable = True
    for L in base.layers[:-50]: 
        L.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    # MC Dropout: training=True keeps dropout active during inference
    x = layers.Dropout(0.5)(x, training=True)
    out = layers.Dense(4, activation='softmax')(x)
    
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-4), 
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ========== PART 3: TRAIN ==========
print("\n\n" + "="*60)
print("PART 2: TRAINING MOBILENETV2 WITH MC DROPOUT")
print("="*60)

pre = pre_mob
tr_aug = ImageDataGenerator(
    preprocessing_function=pre, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
    horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)

tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='training', seed=42)
val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)
te = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)

mc_model = build_mc_model()

cb = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

mc_model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)

# Standard eval
te.reset(); loss, acc = mc_model.evaluate(te, verbose=0)
te.reset(); yp = np.argmax(mc_model.predict(te, verbose=0), axis=1)
yt = te.classes
f1 = f1_score(yt, yp, average='weighted')
print(f"\nMC Model — Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")

# ========== PART 4: UNCERTAINTY ==========
print("\n\n" + "="*60)
print("PART 3: MC DROPOUT UNCERTAINTY ANALYSIS")
print("="*60)

def mc_predict(model, generator, n_iter=50):
    generator.reset()
    all_preds = []
    for _ in range(n_iter):
        generator.reset()
        preds = model.predict(generator, verbose=0)
        all_preds.append(preds)
    all_preds = np.array(all_preds)
    mean_pred = np.mean(all_preds, axis=0)
    entropy = -np.sum(mean_pred * np.log(mean_pred + 1e-10), axis=1)
    return mean_pred, entropy

te.reset()
mean_pred, entropy = mc_predict(mc_model, te, n_iter=50)
y_true = te.classes
y_pred = np.argmax(mean_pred, axis=1)
correct = (y_true == y_pred)

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1
axes[0].hist(entropy[correct], bins=25, alpha=0.7, label='Correct', color='green', density=True)
axes[0].hist(entropy[~correct], bins=25, alpha=0.7, label='Incorrect', color='red', density=True)
axes[0].set_xlabel('Predictive Entropy (nats)')
axes[0].set_ylabel('Density')
axes[0].set_title('Uncertainty: Correct vs Incorrect Predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2
class_entropy = [entropy[y_true == i] for i in range(4)]
bp = axes[1].boxplot(class_entropy, labels=class_names, patch_artist=True)
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Predictive Entropy (nats)')
axes[1].set_title('Uncertainty Distribution per Class')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/uncertainty_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("UNCERTAINTY STATISTICS")
print("="*60)
print(f"Mean entropy (correct):   {np.mean(entropy[correct]):.4f}")
print(f"Mean entropy (incorrect): {np.mean(entropy[~correct]):.4f}")
for i, name in enumerate(class_names):
    mask = y_true == i
    print(f"{name:<12} mean entropy = {np.mean(entropy[mask]):.4f}")

print("\n✅ TAB 2 COMPLETE. Download uncertainty_analysis.png")

PART 1: INFERENCE BENCHMARKING

Benchmarking mobilenetv2...


/tmp/ipykernel_58/1634487549.py:31: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
I0000 00:00:1785601031.961929      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1785601031.968205      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Params: 2,427,588
  Inference: 147.54 ± 5.15 ms
  Throughput: 6.8 img/s

Benchmarking resnet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
  Params: 23,858,692
  Inference: 219.43 ± 6.15 ms
  Throughput: 4.6 img/s

Benchmarking efficientnetb0...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
  Params: 4,219,175
  Inference: 283.50 ± 10.33 ms
  Throughput: 3.5 img/s

BENCHMARK TABLE (for paper)
Model              Params (M)     Inference (ms)     Throughput     
------------------------------------------------------------
mobilenetv2        2.43           147.54             6.8            
resnet50           23.86          219.43             4.6            
efficientnetb0     4.22           283.50             3.5            


PART 2: TRAINING MOBILENETV2 WITH MC DROPOUT
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.


/tmp/ipykernel_58/1634487549.py:97: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)


Epoch 1/30


2026-08-01 16:18:53.998073: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-08-01 16:18:54.136501: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1785601141.225563     130 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


140/140 ━━━━━━━━━━━━━━━━━━━━ 134s 744ms/step - accuracy: 0.7451 - loss: 0.7360 - val_accuracy: 0.6170 - val_loss: 1.1553 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 533ms/step - accuracy: 0.8821 - loss: 0.3386 - val_accuracy: 0.6955 - val_loss: 1.0146 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 541ms/step - accuracy: 0.9210 - loss: 0.2225 - val_accuracy: 0.7857 - val_loss: 0.6828 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 537ms/step - accuracy: 0.9362 - loss: 0.1831 - val_accuracy: 0.8973 - val_loss: 0.3514 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 539ms/step - accuracy: 0.9513 - loss: 0.1365 - val_accuracy: 0.8991 - val_loss: 0.3050 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 535ms/step - accuracy: 0.9632 - loss: 0.1121 - val_accuracy: 0.9304 - val_loss: 0.2130 - learning_rate: 1.0000e-04
Epoch 7/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 75s 538ms/step -

In [ ]:
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as pre
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import seaborn as sns

BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

# Build MC Dropout model
tf.random.set_seed(42); np.random.seed(42)
inp = layers.Input(shape=(224, 224, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
base.trainable = True
for L in base.layers[:-50]: L.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x, training=True)  # MC Dropout
out = layers.Dense(4, activation='softmax')(x)
model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# Data
tr_aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
    horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='training', seed=42)
val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)
te = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
    TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)

# Train
cb = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]
model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)

# Eval
te.reset(); loss, acc = model.evaluate(te, verbose=0)
te.reset(); yp = np.argmax(model.predict(te, verbose=0), axis=1)
yt = te.classes
print(f"\nMC Model — Accuracy: {acc*100:.2f}% | F1: {f1_score(yt, yp, average='weighted')*100:.2f}%")

# Uncertainty (MC Dropout inference)
print("\nRunning MC Dropout uncertainty...")
te.reset()
all_preds = []
for _ in range(50):
    te.reset()
    all_preds.append(model.predict(te, verbose=0))
all_preds = np.array(all_preds)
mean_pred = np.mean(all_preds, axis=0)
entropy = -np.sum(mean_pred * np.log(mean_pred + 1e-10), axis=1)
y_true = te.classes
y_pred = np.argmax(mean_pred, axis=1)
correct = (y_true == y_pred)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(entropy[correct], bins=25, alpha=0.7, label='Correct', color='green', density=True)
axes[0].hist(entropy[~correct], bins=25, alpha=0.7, label='Incorrect', color='red', density=True)
axes[0].set_xlabel('Predictive Entropy'); axes[0].set_ylabel('Density')
axes[0].set_title('Correct vs Incorrect'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
class_entropy = [entropy[y_true == i] for i in range(4)]
bp = axes[1].boxplot(class_entropy, labels=class_names, patch_artist=True)
for patch, color in zip(bp['boxes'], ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Predictive Entropy'); axes[1].set_title('Per-Class Uncertainty')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/uncertainty_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Stats
print(f"\nMean entropy (correct): {np.mean(entropy[correct]):.4f}")
print(f"Mean entropy (incorrect): {np.mean(entropy[~correct]):.4f}")
for i, name in enumerate(class_names):
    print(f"{name}: {np.mean(entropy[y_true == i]):.4f}")

print("\n✅ DONE. Download uncertainty_analysis.png")

/tmp/ipykernel_58/1324157748.py:18: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)


Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 102s 583ms/step - accuracy: 0.7478 - loss: 0.7207 - val_accuracy: 0.7723 - val_loss: 0.5693 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 77s 547ms/step - accuracy: 0.8815 - loss: 0.3457 - val_accuracy: 0.7955 - val_loss: 0.5883 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 77s 551ms/step - accuracy: 0.9098 - loss: 0.2403 - val_accuracy: 0.8875 - val_loss: 0.3307 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 546ms/step - accuracy: 0.9462 - loss: 0.1568 - val_accuracy: 0.8929 - val_loss: 0.3203 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 547ms/step - accuracy: 0.9433 - loss: 0.1452 - val_accuracy: 0.8857 - val_loss: 0.3841 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 76s 544ms/step - accuracy: 0.9551 - lo